# Reprodução — supervised_detection_some_ip (Alkhatib, LSTM binário) na T4

Treina/avalia o **LSTM binário** (58 features, janela 128) do Alkhatib numa **GPU T4** do Colab.

Clona a **nossa versão já corrigida** (`GuilhermeFrick/supervised_detection_someip_colab`) —
com `output_dir` absoluto, modo `predict`, etc. — baixa o dataset, organiza em `train/valid/test`
e roda. **Execute as células em ordem** (o `predict` depende do `train` ter salvo o `best_model.pt`).

> **Escopo (honesto):** reproduz o detector **binário deles nos dados deles**. Não habilita testar
> no nosso tráfego gerado (faltaria o extrator cru→58 features, não publicado).

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SEM GPU — Runtime>Change runtime type>T4')

In [ ]:
# 1) Clona a NOSSA versão já corrigida (não o original)
%cd /content
!rm -rf supervised_detection_someip_colab
!git clone -q https://github.com/GuilhermeFrick/supervised_detection_someip_colab
REPO = '/content/supervised_detection_someip_colab'
print('clonado em', REPO)

In [ ]:
# (opcional) stride: 64 = mais leve; 16 = mais janelas/treino. Ajuste se quiser.
f = f'{REPO}/network_configuration_1/someip_lstm.py'
s = open(f).read().replace('options["stride"] = 64', 'options["stride"] = 16')
open(f, 'w').write(s)
print('stride = 16')

In [ ]:
# 2) Baixa o dataset do Dropbox e descompacta
%cd /content
!wget -q -O dataset.zip "https://www.dropbox.com/sh/k5jnnplxb7ptw5b/AAAbS0N5RkVazV-7DG1MvEaZa?dl=1"
!rm -rf raw_data && mkdir -p raw_data && (cd raw_data && unzip -q ../dataset.zip)
import glob
print('amostras encontradas:', len(glob.glob('/content/raw_data/**/*_x.pickle', recursive=True)))

In [ ]:
# 3) Organiza os pickles em train/valid/test (70/15/15)
import os, glob, shutil
dst = f'{REPO}/output/network_configuration_1/data'
for sp in ('train', 'valid', 'test'):
    os.makedirs(f'{dst}/{sp}', exist_ok=True)
xs = sorted(glob.glob('/content/raw_data/**/*_x.pickle', recursive=True))
moved = {'train': 0, 'valid': 0, 'test': 0}
for i, xp in enumerate(xs):
    yp = xp.replace('_x.pickle', '_y.pickle')
    if not os.path.exists(yp):
        continue
    sp = 'train' if i % 5 < 3 else ('valid' if i % 5 == 3 else 'test')
    for p in (xp, yp):
        shutil.copy(p, f'{dst}/{sp}/{os.path.basename(p)}')
    moved[sp] += 1
print('pares por split:', moved)

In [ ]:
# 4) TREINA (GPU). Early-stop em 10 épocas sem melhora; salva o melhor best_model.pt.
%cd /content/supervised_detection_someip_colab/network_configuration_1
!python someip_lstm.py train

In [ ]:
# 5) Confere que o modelo foi salvo ANTES de avaliar
import os
mp = f'{REPO}/output/network_configuration_1/lstm/best_model.pt'
assert os.path.exists(mp), 'best_model.pt não existe — rode a célula de TREINO até o fim primeiro.'
print('OK, modelo salvo:', mp)

In [ ]:
# 6) AVALIA (predict) — varre limiares e reporta F1 / Recall / Precision
%cd /content/supervised_detection_someip_colab/network_configuration_1
!python someip_lstm.py predict

## Notas

- **Sempre rode TREINO (célula 4) antes do PREDICT (célula 6).** O `predict` carrega o
  `best_model.pt` salvo pelo treino — se não existir, dá `FileNotFoundError`.
- **Early stopping:** treina até 200 épocas, mas para após 10 épocas sem melhora na validação
  (e usa o *melhor* checkpoint, não o último).
- Outros modelos: troque `someip_lstm.py` por `someip_rnn.py` / `someip_transformer.py` / `someip_mlp.py`.
- **Escopo:** reprodução do binário do Alkhatib **nos dados dele**; não substitui o Experimento A
  no nosso tráfego gerado.